In [1]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv
from pathlib import Path

notebook_dir = Path(os.getcwd())
root_dir = notebook_dir.parents[0] if len(notebook_dir.parents) > 0 else notebook_dir
env_path = root_dir / ".env"

load_dotenv(dotenv_path=env_path)

EIA_API_KEY = os.getenv("EIA_API_KEY") or os.getenv("Crude_oil_key")
print("Chemin du .env cherché :", env_path.resolve())
print("Ma clé API chargée :", EIA_API_KEY)

if not EIA_API_KEY:
    raise ValueError("Clé API introuvable. Vérifiez votre fichier .env")

url = "https://api.eia.gov/v2/petroleum/pri/gnd/data/"
params = {
    "api_key": EIA_API_KEY,
    "frequency": "weekly",
    "data[0]": "value",
    "sort[0][column]": "period",
    "sort[0][direction]": "desc",
    "length": 100
}

response = requests.get(url, params=params, timeout=30)
response.raise_for_status()
records = response.json()['response']['data']

df_raw = pd.DataFrame(records)
print(df_raw.info())

df_clean = pd.DataFrame()
df_clean['date'] = pd.to_datetime(df_raw['period']).dt.date
df_clean['product_type'] = df_raw.get('product-name', df_raw.get('product')).astype(str)
df_clean['region'] = df_raw.get('area-name', 'US').astype(str)
df_clean['price_usd_per_gallon'] = pd.to_numeric(df_raw['value'], errors='coerce')

if 'process-desc' in df_raw.columns:
    df_clean['grade'] = df_raw['process-desc'].astype(str)
elif 'process_desc' in df_raw.columns:
    df_clean['grade'] = df_raw['process_desc'].astype(str)
else:
    df_clean['grade'] = df_raw.get('process-name', 'Retail').astype(str)

df_clean['source'] = 'EIA'
df_clean = df_clean.dropna(subset=['price_usd_per_gallon'])
df_clean = df_clean[df_clean['price_usd_per_gallon'] > 0]

df_clean = df_clean.drop_duplicates(subset=['date', 'product_type', 'region', 'grade', 'source'])
print(df_clean.head())

Chemin du .env cherché : C:\Users\user\Desktop\Gasoil_Intelligence\scri\scripts\.env
Ma clé API chargée : Ywlv9ChPz0AzigFBT9lGWsrRjwyOafGJlYD9QLvE
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   period              100 non-null    object
 1   duoarea             100 non-null    object
 2   area-name           100 non-null    object
 3   product             100 non-null    object
 4   product-name        100 non-null    object
 5   process             100 non-null    object
 6   process-name        100 non-null    object
 7   series              100 non-null    object
 8   series-description  100 non-null    object
 9   value               99 non-null     object
 10  units               100 non-null    object
dtypes: object(11)
memory usage: 8.7+ KB
None
         date                       product_type   region  \
0  2026-08-03  No 